# 00 — Colab Setup & Full-Pipeline Run

Run this notebook in [Google Colab](https://colab.research.google.com) (Runtime → Run all, or Cell → Run all)
to reproduce the **entire UK wheat forecasting pipeline** from scratch:

| Step | Stage | Notebook |
|------|-------|----------|
| 01 | Data Acquisition | `notebooks/01_Data_Acquisition.ipynb` |
| 02 | Modelling Table | `notebooks/02_Modelling_Table.ipynb` |
| 03 | EDA | `notebooks/03_EDA.ipynb` |
| 04 | Feature Engineering | `notebooks/04_Feature_Engineering.ipynb` |
| 05 | Model (CV, DM tests, PIs, oracle, verify) | `notebooks/05_Model.ipynb` |

Everything the pipeline needs — frozen raw data, canonical modelling table, and
expected thesis outputs — is committed to this repository, so the run works
**fully offline** once cloned (no downloads from Met Office required).


## 1. Clone the repository

Clones this repo into `/content/uk_wheat_pipeline` and moves into it. All
subsequent cells run with that directory as the working directory.


In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/Dickuta/msc-uk-wheat-forecast.git"
PROJECT_DIR = "/content/uk_wheat_pipeline"

os.makedirs("/content", exist_ok=True)
if not os.path.isdir(PROJECT_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, PROJECT_DIR],
        check=True,
    )
    print("Cloned repository to", PROJECT_DIR)
else:
    print("Repository already present at", PROJECT_DIR)

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

## 2. Install dependencies

Installs the pinned versions from `requirements.txt` (numpy, pandas, scipy,
scikit-learn, statsmodels, matplotlib, seaborn, requests, prophet, xgboost).


In [ ]:
import subprocess
import sys

print("Installing project dependencies from requirements.txt ...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)
print("Dependencies installed.")

## 3. Verify the data bundle

The raw weather files, the modelling table and the expected thesis outputs are
committed to the repo. Stage 01 detects the frozen raw files and skips the
network. This cell just confirms the data tree is intact.


In [ ]:
import os

from pathlib import Path

checks = {
    "data/raw/met_office_Tmean_UK.txt": "raw temperature source",
    "data/raw/met_office_Rainfall_UK.txt": "raw rainfall source",
    "data/raw/manifest.csv": "provenance manifest",
    "data/processed/uk_wheat_modelling_table_1980_2024.csv": "modelling table",
    "data/expected/model_comparison_results_corrected.csv": "expected results",
}
for path, label in checks.items():
    ok = os.path.isfile(path)
    print(f"{'OK  ' if ok else 'MISS'} {label:36s} {path}")
    if not ok:
        raise FileNotFoundError(f"Expected committed data file is missing: {path}")

raw_size = sum(p.stat().st_size for p in Path("data/raw").glob("*"))
print(f"\nCommitted data/raw bundle: {raw_size / 1024:.1f} KiB")

## 4. Run all five stages

Executes `notebooks/01` … `notebooks/05` in order, headlessly, via
`nbconvert`. Each notebook is re-saved in place with its outputs, so the
`notebooks/` copies on the VM hold the executed results ready for download.

The run takes roughly **10–20 minutes** (stage 05 does the full
expanding-window CV across 8 models). If a stage ever raises, execution stops
and this cell reports which notebook failed.


In [ ]:
import subprocess

notebooks = [
    "notebooks/01_Data_Acquisition.ipynb",
    "notebooks/02_Modelling_Table.ipynb",
    "notebooks/03_EDA.ipynb",
    "notebooks/04_Feature_Engineering.ipynb",
    "notebooks/05_Model.ipynb",
]

for nb in notebooks:
    print(f"\n=== Executing {nb} ===")
    subprocess.run(
        ["jupyter", "nbconvert", "--to", "notebook", "--execute", "--inplace", nb],
        check=True,
    )
    print(f"    completed: {nb}")

print("\n=== ALL 5 NOTEBOOKS COMPLETED SUCCESSFULLY ===")

## 5. (Optional) Download the executed notebooks

Zips the executed notebooks and any generated outputs, then triggers a browser
download in Colab. Runs only inside Colab; it is skipped if executed locally.


In [ ]:
import os
import subprocess
import zipfile

zip_path = "/content/uk_wheat_pipeline_executed.zip"

with zipfile.ZipFile(zip_path, "w") as z:
    for nb in [
        "notebooks/01_Data_Acquisition.ipynb",
        "notebooks/02_Modelling_Table.ipynb",
        "notebooks/03_EDA.ipynb",
        "notebooks/04_Feature_Engineering.ipynb",
        "notebooks/05_Model.ipynb",
    ]:
        z.write(nb)
    for f in sorted(os.listdir("data/outputs")):
        z.write(os.path.join("data/outputs", f))
    z.write("data/outputs/decision_guide.md")

print("Bundled executed notebooks + outputs into", zip_path)

try:
    from google.colab import files

    files.download(zip_path)
    print("Download started in your browser.")
except ImportError:
    print("Not running inside Colab - skipping browser download.")